In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import pyquist as pq
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the threshold $T$ and the ratio $R$. On the left is the compressor's
static curve (red), which maps each input level to an output level. On the
right are the opening of Beethoven's Fifth Symphony (blue) and the compressed
version (purple), scaled back to the same peak with makeup gain. The readout
measures how far apart the loud and quiet passages are, before and after. The
audio card plays the compressed version.

In [ ]:
# hide
# autorun
T0, R0 = -30.0, 4.0                 # the chapter's example

PURPLE = "#6E3B87"
beethoven = pq.Audio.from_file("../assets/beethoven.wav").as_mono()
SR = beethoven.sample_rate
X = np.asarray(beethoven.samples).reshape(-1)
X = X / np.abs(X).max()
N_H = SR // 100                     # 100 frames per second
N_F = 4 * N_H
RATIOS = [("1", 1.0), ("1.5", 1.5), ("2", 2.0), ("3", 3.0), ("4", 4.0),
          ("6", 6.0), ("10", 10.0), ("20", 20.0), ("∞", float("inf"))]

# the RMS level of every frame, in dB (Chapter 10 frames, vectorized)
_cs = np.concatenate([[0.0], np.cumsum(X ** 2)])
_starts = np.arange(0, len(X) - N_F + 1, N_H)
LEVEL = 10 * np.log10(np.maximum((_cs[_starts + N_F] - _cs[_starts]) / N_F, 1e-12))
CENTERS = _starts + N_F // 2
LOUD = LEVEL > -70                  # ignore the silence for the spread


def static_curve(L, T, R):
    return np.where(L > T, T + (L - T) / R, L)


def gain_per_sample(T, R, LEVEL=LEVEL, CENTERS=CENTERS, N=len(X)):
    g_db = static_curve(LEVEL, T, R) - LEVEL
    return 10 ** (np.interp(np.arange(N), CENTERS, g_db) / 20)


def envelope(x, n_blocks=700):
    step = max(1, len(x) // n_blocks)
    n = len(x) // step * step
    blocks = x[:n].reshape(-1, step)
    t = (np.arange(n // step) + 0.5) * step / SR
    return np.repeat(t, 2), np.column_stack([blocks.min(1), blocks.max(1)]).ravel()


def compressed(T, R, X=X, gain_per_sample=gain_per_sample):
    y = X * gain_per_sample(T, R)
    return y / np.abs(y).max()      # makeup gain: back to the same peak


LIN = np.linspace(-80.0, 0.0, 321)
T_ENV, E_IN = envelope(X)
# the sample where each block of the envelope reaches its min and its max
_step = len(X) // 700
_blocks = X[:len(X) // _step * _step].reshape(-1, _step)
_base = np.arange(len(_blocks)) * _step
PEAKS = np.column_stack([_base + _blocks.argmin(1), _base + _blocks.argmax(1)]).ravel()


def compressed_envelope(T, R, X=X, PEAKS=PEAKS, LEVEL=LEVEL, CENTERS=CENTERS):
    """The compressed waveform's min/max envelope, from the samples where the
    original's envelope peaks: the gain changes slowly, so they stay the peaks."""
    g_db = static_curve(LEVEL, T, R) - LEVEL
    e = X[PEAKS] * 10 ** (np.interp(PEAKS, CENTERS, g_db) / 20)
    return e / np.abs(e).max()              # makeup gain, as in compressed()


def figure():
    fig = make_subplots(rows=2, cols=2, column_widths=[0.36, 0.64],
                        specs=[[{"rowspan": 2}, {}], [None, {}]],
                        horizontal_spacing=0.12, vertical_spacing=0.12)
    fig.add_scatter(x=LIN, y=LIN, mode="lines",
                    line=dict(color=IRON, width=1, dash="dash"), row=1, col=1)
    fig.add_scatter(x=[T0, T0], y=[-80, 0], mode="lines",
                    line=dict(color=STEEL, width=1.2, dash="dot"), row=1, col=1)
    fig.add_scatter(x=LIN, y=static_curve(LIN, T0, R0), mode="lines",
                    line=dict(color=RED, width=2.4), row=1, col=1)
    fig.add_scatter(x=T_ENV, y=E_IN, mode="lines", line=dict(color=BLUE, width=1),
                    row=1, col=2)
    fig.add_scatter(x=T_ENV, y=compressed_envelope(T0, R0), mode="lines",
                    line=dict(color=PURPLE, width=1), row=2, col=2)
    fig.update_xaxes(range=[-80, 0], title_text="Input level L<sub>in</sub> (dB)",
                     fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[-80, 0], title_text="Output level L<sub>out</sub> (dB)",
                     fixedrange=True, row=1, col=1)
    fig.update_xaxes(range=[0, len(X) / SR], showticklabels=False,
                     fixedrange=True, row=1, col=2)
    fig.update_yaxes(range=[-1.05, 1.05], title_text="Original",
                     fixedrange=True, row=1, col=2)
    fig.update_xaxes(range=[0, len(X) / SR], title_text="Time (s)",
                     fixedrange=True, row=2, col=2)
    fig.update_yaxes(range=[-1.05, 1.05], title_text="Compressed",
                     fixedrange=True, row=2, col=2)
    fig.update_layout(height=420)
    return fig


def controls(fig):
    T = widgets.FloatSlider(description="Threshold $T$ (dB)", min=-60.0,
                            max=0.0, step=1.0, value=T0, readout_format=".0f")
    R = widgets.SelectionSlider(description="Ratio $R$", options=RATIOS,
                                value=R0)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(T, R, static_curve=static_curve,
               compressed_envelope=compressed_envelope, LIN=LIN, LEVEL=LEVEL,
               LOUD=LOUD, readout=readout):
        with fig.batch_update():
            fig.data[1].x = [T, T]
            fig.data[2].y = static_curve(LIN, T, R)
            fig.data[4].y = compressed_envelope(T, R)
        lin, lout = LEVEL[LOUD], static_curve(LEVEL[LOUD], T, R)
        spread_in = np.percentile(lin, 95) - np.percentile(lin, 5)
        spread_out = np.percentile(lout, 95) - np.percentile(lout, 5)
        readout.value = (
            f"<span style='font-size:0.9em'>loud-to-quiet spread (95th minus 5th "
            f"percentile of the RMS level): {spread_in:.0f} dB &rarr; "
            f"{spread_out:.0f} dB</span>")

    widgets.interactive_output(update, {"T": T, "R": R})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def show_audio(x, SR=SR):
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    def render(compressed=compressed):
        y = 0.5 * compressed(T.value, R.value)
        y[-441:] *= np.linspace(1, 0, 441)
        show_audio(y)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (T, R):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([T, R, readout, out, gate])

icm_plotly.show(figure, controls)